In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

### Load Data

In [2]:
path = "../data/raw/PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(path)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 706.2 MB


### Feature Selection

In [6]:
def perform_feature_engineering(X):
    df_new = X.copy()
    
    # 1. Hour of Day
    df_new['hour_of_day'] = (df_new['step'] % 24).astype('int8')
    
    # 2. Check Balance Orig
    df_new['errorBalanceOrig'] = df_new['newbalanceOrig'] + df_new['amount'] - df_new['oldbalanceOrg']
    
    # 3. Check Balance Dest & Flagging Merchant
    df_new['is_merchant_dest'] = df_new['nameDest'].str.startswith('M').astype('int8')
    df_new['errorBalanceDest'] = np.where(
        df_new['is_merchant_dest'] == 1, 0,
        df_new['oldbalanceDest'] + df_new['amount'] - df_new['newbalanceDest']
    )
    df_new['errorBalanceDest'] = (
        np.sign(df_new['errorBalanceDest']) * np.log1p(np.abs(df_new['errorBalanceDest']))
    )
    df_new['errorBalanceOrig'] = (
        np.sign(df_new['errorBalanceOrig']) * np.log1p(np.abs(df_new['errorBalanceOrig']))
    )

    # 4 Balance Drain Ratio
    df_new['has_zero_orig_balance'] = (df_new['oldbalanceOrg'] == 0).astype('int8')
    df_new['balance_drain_ratio'] = np.where(
        df_new['oldbalanceOrg'] > 0,
        df_new['amount'] / df_new['oldbalanceOrg'],
        0
    )
    df_new['balance_drain_ratio'] = df_new['balance_drain_ratio'].clip(0, 10)

    # 5. Binning Time Segmentation
    bins_time = [-1, 6, 18, 24]
    labels_time = ['Midnight_to_Morning', 'Working_Hours', 'Evening']
    df_new['time_segment'] = pd.cut(df_new['hour_of_day'], bins=bins_time, labels=labels_time)

    # 6. Binning Amount (qcut)
    labels_amount = ['Low_Amount', 'Medium_Amount', 'High_Amount']
    df_new['amount_category'] = pd.qcut(df_new['amount'], q=3, labels=labels_amount)

    # 7. Drop noise & redundan feature
    cols_to_drop = [
        'nameOrig', 'nameDest', 'newbalanceDest', 'newbalanceOrig',
        'step', 'is_merchant_dest', 'isFraud', 'isFlaggedFraud'
    ]
    cols_to_drop = [c for c in cols_to_drop if c in df_new.columns]
    df_new = df_new.drop(columns=cols_to_drop)
    
    return df_new


Justification:
1. Temporal Extraction
The default step feature is linear sequential. Through a modulo operation (step % 24), the data is transformed into a daily temporal cycle representation (0–23). The next binning step into three business segments (Midnight_to_Morning, Working_Hours, Evening) is based on empirical EDA findings showing extreme fraud rate spikes (>20%) in the early morning (off-peak hours). This grouping helps the model distinguish normal human operational patterns from structured attacks.

2. Check Financial Logic
In the financial system, the ending balance must be mathematically balanced against the beginning balance minus/plus the transaction amount ($Saldo_{akhir} = Saldo_{awal} \pm Amount$). The creation of an error balance feature on the sender and receiver sides aims to explicitly capture this logical deviation. A value of $\neq 0$ in this feature is a strong anomaly (hard signal) indicating account manipulation, system bypass, or indication of money laundering (layering).

3. Handling Merchant Limitations
The dataset has a limitation where all transactions with a Merchant destination do not track balance changes, so by design the destination balance value is set to 0. If calculated directly, this would create a large false anomaly in the errorBalanceDest. The is_merchant_dest flag is used as a filter to force the merchant error balance value to 0, so that the clustering algorithm is not distorted by this data recording error.

4. Data Transformation Using Logarithms
The distribution of error balance values ​​is severely right-skewed and bimodal due to the large number of 0 values ​​(merchant effect). If directly included in the distance weighting (scaling), these extreme outliers would dominate the calculation and distort the inter-quartile range (IQR). Using the Signed Log Transform ($sign(x) \times \ln(|x| + 1)$) is the absolute solution for drastically reducing extreme skewness, while maintaining the original transaction direction/sign (+/-) without producing undefined/error values ​​at 0.

5. Account Drainage Ratio
Fraudsters have a tendency to completely drain victims' accounts (complete account draining). The balance_drain_ratio feature captures the intensity of this draining. The has_zero_orig_balance feature is then used to set a ratio of 0 for accounts with no initial balance. Furthermore, a limit on extreme values ​​(.clip(0, 10)) is applied to control the variance caused by the division of very small floating-point values.

6. Amount Category Discretization
Some association algorithms or rules (such as Apriori) require purely categorical input data. Transforming the numeric amounts into three tertile levels using pd.qcut ensures a perfectly balanced distribution of the data frequencies (33.33% each). This maximizes the entropy value of feature information and prevents algorithm bias toward certain nominal classes.

7. Eliminating redundancy and data leakage
nameOrig & nameDest are discarded because their high cardinality provides no analytical value but only increases computational burden. newbalanceOrig & newbalanceDest are discarded because their information has already been extracted and is more informatively represented by the error balance feature (avoiding perfect multicollinearity). isFraud & isFlaggedFraud must be discarded because they are target labels that must not leak (data leakage) into the unsupervised learning/data mining process.

In [4]:
def select_segmentation_features(X):
    return X.drop(columns=['time_segment', 'amount_category'])

def select_pattern_features(X):
    return X[['type', 'time_segment', 'amount_category']]

### Pipeline preprocessing

In [ ]:
nums_cols = ['amount', 'oldbalanceOrg', 'oldbalanceDest', 'errorBalanceOrig', 'errorBalanceDest', 'balance_drain_ratio', 'hour_of_day']
cat_cols = ['type']
binary_cols = ['has_zero_orig_balance']

preprocessor_seg = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), nums_cols),
        ('cat', OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore'), cat_cols),
        ('bin', 'passthrough', binary_cols)
    ],
    remainder='drop'
)

segmentation_pipeline = Pipeline([
    ('core_eng', FunctionTransformer(perform_feature_engineering)),
    ('selector', FunctionTransformer(select_segmentation_features)),
    ('preprocessing', preprocessor_seg)
])

pattern_pipeline = Pipeline([
    ('core_eng', FunctionTransformer(perform_feature_engineering)),
    ('selector', FunctionTransformer(select_pattern_features))
])

In [9]:
data_seg_array = segmentation_pipeline.fit_transform(df)

In [12]:
# 1. Clustering
feature_names = (
    nums_cols + 
    list(segmentation_pipeline.named_steps['preprocessing'].transformers_[1][1].get_feature_names_out(cat_cols)) +
    binary_cols
)
df_clustering_final = pd.DataFrame(data_seg_array, columns=feature_names)
df_clustering_final.to_parquet('../data/processed/real_used/data_phase2_clustering.parquet', index=False)
print("shape clustering: ", df_clustering_final.shape)

# 2. Apriori
df_rules_final = pattern_pipeline.fit_transform(df)
df_rules_final.to_parquet('../data/processed/real_used/data_phase3_rules.parquet', index=False)
print("shape apriori: ", df_rules_final.shape)

shape clustering:  (6362620, 13)
shape apriori:  (6362620, 3)


In [13]:
df_clustering_final.head()

,amount,oldbalanceOrg,oldbalanceDest,errorBalanceOrig,errorBalanceDest,balance_drain_ratio,hour_of_day,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,has_zero_orig_balance
0,-0.332932,1.452991,-0.140722,-2.510382,0.000000,-0.008550,-2.142857,0.0,0.0,0.0,1.0,0.0,0.0
1,-0.373762,0.065610,-0.140722,-2.510382,0.000000,0.004922,-2.142857,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.382380,-0.130708,-0.140722,-2.510382,522.998356,0.415954,-2.142857,0.0,0.0,0.0,0.0,1.0,0.0
3,-0.382380,-0.130708,-0.118260,-2.510382,1001.922718,0.415954,-2.142857,0.0,1.0,0.0,0.0,0.0,0.0
4,-0.323571,0.254820,-0.140722,-2.510382,0.000000,0.091907,-2.142857,0.0,0.0,0.0,1.0,0.0,0.0


In [14]:
df_rules_final.head()

,type,time_segment,amount_category
0,PAYMENT,Midnight_to_Morning,Low_Amount
1,PAYMENT,Midnight_to_Morning,Low_Amount
2,TRANSFER,Midnight_to_Morning,Low_Amount
3,CASH_OUT,Midnight_to_Morning,Low_Amount
4,PAYMENT,Midnight_to_Morning,Low_Amount
